In [10]:
import os, sys
import numpy as np
import rasterio
from pystac_client import Client

def build_unet_input(bbox, impervious_path=None, canopy_path=None, out_tif="unet_input.tif"):
    catalog = Client.open("https://catalogue.dataspace.copernicus.eu/stac/v1/")

    item = next(catalog.search(
        bbox=bbox,
        collections=["sentinel-2-l2a"],
        query={"eo:cloud_cover": {"lt": 20}},
        limit=1
    ).items(), None)

    # === Load 10m bands ===
    red_href = item.assets["B04_10m"].href
    nir_href = item.assets["B08_10m"].href
    blue_href = item.assets["B02_10m"].href
    green_href = item.assets["B03_10m"].href

    with rasterio.open(red_href) as src:
        red = src.read(1).astype("float32") / 10000
        meta = src.meta.copy()

    with rasterio.open(nir_href) as src:
        nir = src.read(1).astype("float32") / 10000

    with rasterio.open(blue_href) as src:
        blue = src.read(1).astype("float32") / 10000

    with rasterio.open(green_href) as src:
        green = src.read(1).astype("float32") / 10000

    # === NDVI ===
    ndvi = (nir - red) / (nir + red + 1e-6)

    channels = [red, nir, blue, green, ndvi]

    # === Optional external predictors ===
    if impervious_path:
        with rasterio.open(impervious_path) as src:
            channels.append(src.read(1).astype("float32"))

    if canopy_path:
        with rasterio.open(canopy_path) as src:
            channels.append(src.read(1).astype("float32"))

    stack = np.stack(channels, axis=0)

    meta.update(count=len(channels), dtype="float32")

    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(stack)

    print(f"Saved U-Net input stack with {len(channels)} channels → {out_tif}")
    return out_tif


In [11]:
area = "Caen"
# Define bbox in WGS84
lon_wgs84, lat_wgs84 = -0.3, 49.2  # Caen
box_size = 0.01 # degrees
#####

lon_min_wgs84, lat_min_wgs84 = lon_wgs84 - box_size, lat_wgs84 - box_size
lon_max_wgs84, lat_max_wgs84 = lon_wgs84 + box_size, lat_wgs84 + box_size
bbox_wgs84 = {"lon_min": lon_min_wgs84, "lat_min": lat_min_wgs84, "lon_max": lon_max_wgs84, "lat_max": lat_max_wgs84}
bbox = [bbox_wgs84["lon_min"], bbox_wgs84["lat_min"], bbox_wgs84["lon_max"], bbox_wgs84["lat_max"]]

toto = build_unet_input(bbox)


APIError: {"detail":"Not Found"}

In [16]:
from pystac_client import Client

catalog = Client.open("https://catalogue.dataspace.copernicus.eu/api/stac/v1")
print(catalog)

APIError: <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</center>
</body>
</html>


In [1]:
client_id = "sh-488d557f-1a38-4599-83e0-641dd6aac7c8"

client_secret = "6scM5JB6aGHkZjQJUXna8wRJLBKDcSql"

import requests

def get_cdse_token(client_id, client_secret):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

    data = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret
    }

    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()["access_token"]

token = get_cdse_token(client_id, client_secret)
print(token[:50], "...")

eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6IC ...


In [2]:
import time
from pystac_client import Client
from requests.exceptions import RequestException

for i in range(10):
    try:
        catalog = Client.open(
            "https://sh.dataspace.copernicus.eu/api/stac/v1",
            headers={"Authorization": f"Bearer {token}"}
        )
        print("Connected successfully!")
        break
    except Exception as e:
        print(f"Attempt {i+1} failed: {e}")
        time.sleep(5 * (i+1))  # exponential backoff

Attempt 1 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 2 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 3 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 4 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 5 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 6 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 7 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


Attempt 8 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
<

KeyboardInterrupt: 

In [5]:
from pystac_client import Client
import time

# Tes identifiants CDSE

client_id = "sh-488d557f-1a38-4599-83e0-641dd6aac7c8"
client_secret = "6scM5JB6aGHkZjQJUXna8wRJLBKDcSql"

import requests

def get_cdse_token(client_id, client_secret):
    url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret
    }
    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()["access_token"]

token = get_cdse_token(client_id, client_secret)

# Fonction pour ouvrir un STAC catalog avec retry et fallback

def open_stac_catalog():
    cdse_url = "https://sh.dataspace.copernicus.eu/api/stac/v1"
    public_url = "https://earth-search.aws.element84.com/v0"


    # Essayer CDSE d'abord
    for i in range(5):
        try:
            catalog = Client.open(
                cdse_url,
                headers={"Authorization": f"Bearer {token}"}
            )
            print("Connecté au CDSE STAC API ✅")
            return catalog
        except Exception as e:
            print(f"CDSE attempt {i+1} failed: {e}")
            time.sleep(5 * (i+1))

    # Si CDSE indisponible, utiliser un endpoint public
    print("CDSE indisponible, bascule sur endpoint public AWS STAC 🌐")
    catalog = Client.open(public_url)
    return catalog

catalog = open_stac_catalog()

# Exemple : rechercher des images Sentinel-2 sur une zone et période

search = catalog.search(
collections=["sentinel-s2-l2a"],
bbox=[2.2945, 48.8584, 2.2955, 48.8594],  # petit carré autour de la Tour Eiffel
datetime="2023-01-01/2023-01-31",
limit=5
)

items = list(search.get_items())
print(f"Nombre d'items récupérés : {len(items)}")
for item in items:
    (item.id, item.datetime)


CDSE attempt 1 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


CDSE attempt 2 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


CDSE attempt 3 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


CDSE attempt 4 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


CDSE attempt 5 failed: <html><body><h1>503 Service Unavailable</h1>
No server is available to handle this request.
</body></html>


CDSE indisponible, bascule sur endpoint public AWS STAC 🌐


/home/renauld/.local/lib/python3.12/site-packages/pystac_client/client.py:191: NoConformsTo: Server does not advertise any conformance classes.
  warnings.warn(NoConformsTo())


DoesNotConformTo: Server does not conform to ITEM_SEARCH, There is no fallback option available for search.

In [6]:
from pystac_client import Client

aws_url = "https://earth-search.aws.element84.com/v0"
catalog = Client.open(aws_url)

# AWS n'a pas ITEM_SEARCH complet, on récupère directement les items
items = list(catalog.get_all_items())

# Filtrer par collection et date
sentinel_items = [
    item for item in items
    if item.collection_id == "sentinel-s2-l2a" and "2023-01" in item.datetime.isoformat()
]

print(f"Nombre d'items filtrés : {len(sentinel_items)}")
for item in sentinel_items[:5]:
    print(item.id, item.datetime)

/home/renauld/.local/lib/python3.12/site-packages/pystac_client/client.py:475: FallbackToPystac: Falling back to pystac. This might be slow.
  self._warn_about_fallback("ITEM_SEARCH")


Nombre d'items filtrés : 0


/home/renauld/.local/lib/python3.12/site-packages/pystac_client/collection_client.py:149: FallbackToPystac: Falling back to pystac. This might be slow.
  root._warn_about_fallback("ITEM_SEARCH")
